# MDT + DDM, from the ground up

A self-contained explanation of the two papers' machinery, the key theorem (**Eckart–Young**), and every
experimental finding, assuming no prior reading. Math renders in Jupyter (a terminal shows raw `$$`).
Run the code cells top to bottom — each one is self-contained given the ones above it.

**Roadmap**
- **Part I — From data to the embedding:** kernel → diffusion → diffusion maps → SVD → Eckart–Young → MDT (kernel + fusion).
- **Part II — Adding a neural network (DDM):** Gram-matching, the ceiling, and *imit-Gram vs end-to-end*.
- **Part III — The findings (and why):** imit≠fixed, bigger-dim-hurts, sparse-vs-dense, the Internal-Quality index.
- **Part IV — One root cause:** the flat spectrum.


# Part I — From data to the embedding

## 1. The problem
We have $N$ data points $x_1,\dots,x_N$, each a vector (an image = a list of pixel values, etc.). Goal:
**cluster them** with no labels, or place them in a low-dimensional space where similar points sit close.


## 2. Similarity $\to$ kernel $\to$ random walk
A Gaussian **kernel** measures pairwise similarity:

$$ K_{ij} \;=\; \exp\!\left(-\,\frac{\lVert x_i - x_j\rVert^2}{\sigma^2}\right)\ \in(0,1], $$

with bandwidth $\sigma$ ("how close counts as close"). Row-normalise to get a **transition matrix**
(a random walk on the data):

$$ P = D^{-1}K,\qquad D_{ii}=\sum_j K_{ij},\qquad P_{ij}=\Pr(i\to j\ \text{in one hop}). $$

The **kNN** version keeps only each point's $k$ nearest neighbours (a *sparse* $K$).


## 3. Diffusion: the operator carries the cluster structure
Taking $t$ steps, $P^{t}$, is **diffusion**. Two points in the same cluster reach a similar set of places
(walks mix inside a cluster), so their rows of $P^t$ look alike; points in different clusters do not. This
is the **diffusion distance** — it follows the shape of the data, unlike raw Euclidean distance.


## 4. Diffusion maps: operator $\to$ coordinates
We want short coordinates $\psi(x_i)$ with $\lVert\psi(x_i)-\psi(x_j)\rVert\approx$ diffusion distance.
The classical result: they are the **top eigenvectors / singular vectors of the operator, scaled by its
eigen/singular values**. Keep the top $k$, run k-means. So the whole task is: **decompose the operator,
keep the top-$k$ pieces.**


## 5. Eigendecomposition and SVD
**Eigendecomposition** (symmetric $M$): $\,M=\sum_i\lambda_i u_iu_i^\top$ — orthonormal **eigenvectors**
$u_i$, **eigenvalues** $\lambda_i$. **SVD** (any matrix): $\,W=\sum_i\sigma_i u_i v_i^\top$ — **singular
values** $\sigma_i\ge0$ (large$\to$small), left/right singular vectors $u_i,v_i$. We use the SVD because
MDT's operator is a product of row-normalised matrices — **not symmetric** — so it has no clean
eigendecomposition, but the SVD always exists. **Truncated SVD** keeps the top $k$ terms.


## 6. The Eckart–Young(–Mirsky) theorem  ⭐
Among **all** matrices of rank $\le k$, the truncated SVD $W_k$ is the **closest** to $W$:

$$ W_k=\arg\min_{\operatorname{rank}(M)\le k}\lVert W-M\rVert,\qquad
   \lVert W-W_k\rVert_F^2=\sum_{i>k}\sigma_i^2 . $$

*In words:* the best rank-$k$ summary of a matrix is its top-$k$ singular pieces — nothing rank-$k$ is
closer. It is a **theorem** (1936). Consequence: the truncated SVD is *the best possible* rank-$k$
summary of the operator, so any method aiming for "best rank-$k$ summary" is **capped by the SVD** (the
"ceiling"). The next cell runs the full pipeline and checks the Eckart–Young error identity numerically.


In [ ]:
import numpy as np
from numpy.linalg import svd, norm
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_mutual_info_score as AMI

rng = np.random.default_rng(0)                                     # toy: two 2D blobs
X = np.vstack([rng.normal([0, 0], 0.35, (40, 2)), rng.normal([3, 3], 0.35, (40, 2))])
truth = np.r_[np.zeros(40), np.ones(40)].astype(int)

K = np.exp(-cdist(X, X, "sqeuclidean") / 1.0**2)                   # 2. kernel
P = K / K.sum(1, keepdims=True)                                    # 2. transition
U, s, Vt = svd(P)                                                  # 5. SVD
print("top singular values:", np.round(s[:6], 3))

k = 2
psi = U[:, 1:k+1] * s[1:k+1]                                       # 4. embedding (drop trivial 1st)
Pk = (U[:, :k] * s[:k]) @ Vt[:k]                                   # 6. Eckart-Young check
print(f"Eckart-Young:  ||P-P_k||_F^2 = {norm(P-Pk)**2:.6f}   sum_(i>k) s_i^2 = {(s[k:]**2).sum():.6f}"
      f"   equal? {np.isclose(norm(P-Pk)**2,(s[k:]**2).sum())}")
labels = KMeans(2, n_init=10, random_state=0).fit_predict(psi)
print("clustering AMI vs truth:", round(AMI(truth, labels), 3), " (1.0 = perfect)")


## 7. MDT: multi-view = **kernel** + **fusion** + trajectory
Real objects have several **views** (colour, texture, shape). The operator is built from two ingredients:

**Kernel (per view):** view $v$ $\to$ similarity $\to$ transition,
$\;K_v=\exp(-\lVert\cdot\rVert^2/\sigma_v^2),\ P_v=D_v^{-1}K_v.$

**Fusion (across views):** a **convex mixture** of the views at each step, chained into a length-$t$
**trajectory**:

$$ W_s=\sum_v a_{sv}P_v\quad(a_{sv}\ge0,\ \textstyle\sum_v a_{sv}=1),\qquad W=W_t\cdots W_1 . $$

The **embedding is the truncated SVD of $W$**. These two ingredients are exactly what "fixed MDT" leaves
fixed and what the end-to-end network (Part II) tries to learn:

| ingredient | what it does | fixed MDT | end-to-end network |
|---|---|---|---|
| **kernel** $K_v,P_v$ | view $\to$ transition | fixed Gaussian on raw features | learnable projection $\phi_v$ + bandwidth |
| **fusion** $a_{sv}$ | blend views into one step | fixed / selected trajectory | learned convex weights (softmax) |

The cell shows fusion concretely: different weights $a$ blend two MSRC views into different operators.


In [ ]:
import scipy.io
from scipy.spatial.distance import squareform, pdist
from functools import reduce

def transition(Xview, knn):                                        # a view's kernel -> transition
    D = squareform(pdist(Xview)); n = len(Xview)
    bw = max(row[row > 0].min() for row in D)                      # tight max-of-row-min bandwidth
    Kk = np.exp(-D**2 / bw); Ks = np.zeros_like(Kk)
    for i in range(n):
        nn = np.argsort(D[i])[:knn+1]; Ks[i, nn] = Kk[i, nn]       # keep kNN neighbours
    Ks = (Ks + Ks.T) / 2
    return Ks / Ks.sum(1, keepdims=True)

m = scipy.io.loadmat("/tmp/Multi-view-datasets/MSRC-v5.mat")
Xv = [np.asarray(v).astype(float) for v in m["X"].ravel()]
y = np.asarray(m["y"]).ravel(); k = len(np.unique(y))
knn = max(2, int(np.floor(np.log(len(y)))))
P0, P1 = transition(Xv[0], knn), transition(Xv[1], knn)            # two views' transitions

print("Fusion  W = a0*P0 + a1*P1  ->  SVD embedding  ->  k-means:")
for a0, a1 in [(1.0, 0.0), (0.7, 0.3), (0.5, 0.5), (0.0, 1.0)]:
    W = a0 * P0 + a1 * P1
    U, s, _ = svd(W); E = U[:, 1:k+1] * s[1:k+1]
    print(f"  a=({a0:.1f},{a1:.1f})  AMI = {AMI(y, KMeans(k, n_init=5, random_state=0).fit_predict(E)):.3f}")
print("-> the fusion weights matter: a bad view alone is useless, blended it clusters well.")
print("   FIXED MDT picks the weights (a trajectory); END-TO-END learns them (Part II).")


# Part II — Adding a neural network (DDM)

## 8. DDM: replace the SVD with a trained network
Computing the SVD is fine but *transductive*: a new point gets no coordinates. **Deep Diffusion Maps
(DDM)** trains a network $f_\theta(x)$ to **output** the embedding. It trains via the **Gram matrix**
$FF^\top$ (inner products of the embedding rows): build a target Gram $G$ from the operator, and match it,

$$ \min_\theta\ \lVert F F^\top - G\rVert^2,\qquad F=f_\theta(X). $$

No SVD inside the network — it only pushes inner products toward $G$. (The paper's $\sqrt{\pi}$ weighting
is a technical fix for the trivial constant mode; ignore it for the idea.)


## 9. The ceiling: the network can only *match* the SVD
$F$ is $N\times k$, so $FF^\top$ ranges over **all rank-$k$ PSD matrices**. Minimising
$\lVert FF^\top-G\rVert^2$ asks for the rank-$k$ matrix closest to $G$ — which, by **Eckart–Young**, is
the top-$k$ eigendecomposition of $G$, i.e. the truncated SVD embedding. So the global optimum **is** the
SVD: a perfect network reproduces it and **cannot beat** it. That is the *ceiling*. The cell confirms the
top-$k$ decomposition beats any other rank-$k$ matrix.


In [ ]:
G = P0 @ P0.T                                                     # a symmetric PSD target
w, Q = np.linalg.eigh(G); idx = np.argsort(w)[::-1][:k]
Gk = (Q[:, idx] * w[idx]) @ Q[:, idx].T                            # top-k eigen = Eckart-Young optimum
R = rng.standard_normal((len(G), k)); Grand = R @ R.T; Grand *= norm(Gk) / norm(Grand)
print(f"||G - top_k(G)||     = {norm(G-Gk):.3f}   (Eckart-Young optimum: the SVD)")
print(f"||G - random_rank_k|| = {norm(G-Grand):.3f}   (any other rank-k is >= the optimum)")
print("-> the SVD is the best rank-k target; a Gram-matching network can at best REACH it, never beat it.")


## 10. Two ways to use the network: imit-Gram vs end-to-end — and why end-to-end fails
Both use a network; that is all they share.

| | **imit-Gram** (Scenario 1) | **end-to-end** (Scenario 4) |
|---|---|---|
| operator | **fixed** (a good MDT operator) | **learned** (kernels $\phi_v$ + fusion $a_{sv}$) |
| learns | the **encoder** $f_\theta$ | the **operator/geometry** itself |
| objective | Gram-matching a **fixed** target $G$ | **contrastive** NLL on the operator's entries |
| landscape | clear optimum (the SVD) $\to$ converges | near-uniform logits $\to$ flat $\to$ vanishing gradients |
| result | **matches** fixed MDT | **collapses** below fixed |

**Why end-to-end fails:** (1) the contrastive objective is *misaligned with clustering* — even starting
from an operator that equals fixed MDT, training degrades it ($23/24$ runs, paper Table 8); (2) the
**uniform-logit stall** — the contrastive loss softmaxes each operator row, but a diffused operator's
rows are $\approx 1/N$, so the softmax is $\approx$ uniform, the loss is flat, gradients vanish.
**imit-Gram sidesteps both** by keeping the operator fixed and only reproducing its SVD. The cell shows
the stall: softmax$(W)$ max-probability $\approx 1/N$.


In [ ]:
Pv = [transition(v, knn) for v in Xv]
print(f"uniform baseline 1/N = {1/len(y):.4f}\n")
for t in (1, 2, 4, 6):
    W = reduce(lambda A, B: B @ A, [Pv[i] for i in rng.integers(0, len(Pv), t)])
    ex = np.exp(np.clip(W, -20, 20)); np.fill_diagonal(ex, 0.0)
    sm = ex / ex.sum(1, keepdims=True)
    print(f"  t={t}:  softmax(W) mean row max-prob = {sm.max(1).mean():.4f}   (mean |W_ij| = {np.abs(W).mean():.1e})")
print("\n-> softmax(W) ~ 1/N (uniform) at every t -> contrastive loss FLAT -> gradients vanish -> end-to-end STALLS.")
print("   imit-Gram matches a FIXED operator's SVD (a sharp target) -> converges -> matches fixed MDT.")


### Why imit-Gram *succeeds* — the two fixes

imit-Gram turns "learn a good geometry" into **"copy a fixed, known-good answer"** — that one move repairs both failures.

**Fix 1 — alignment (repairs the misaligned objective).** The target $G$ is built *once* from a fixed good operator; it *is* the inner-product structure of an embedding that already clusters well. Minimising $\lVert FF^\top - G\rVert^2$ means "make your embedding's inner products equal the good one's," whose unique minimiser (up to rotation) is that good embedding. So **lowering the loss $\Leftrightarrow$ approaching the good embedding $\Leftrightarrow$ better clustering** — loss and goal *coincide*, and the target never moves. (The contrastive loss is a proxy that can *worsen* clustering, with a target that moves every step.)

**Fix 2 — a real gradient (repairs the flat plateau).**

- *Contrastive is flat:* it pushes the tiny entries $W_{ij}\approx 1/N$ through $\exp(\cdot)$; $\exp(\text{tiny})\approx 1$ for all of them $\to$ uniform softmax $\to$ the loss barely responds $\to$ gradient $\approx 0$. The $\exp$/softmax **saturates**, and that saturation *is* the plateau.
- *Gram is a bowl:* $\lVert FF^\top - G\rVert^2$ is a least-squares objective, with gradient
$$\nabla_F\,\lVert FF^\top - G\rVert^2 \;=\; 4\,(FF^\top - G)\,F,$$
which is **large whenever $FF^\top \ne G$** — a real slope pointing straight at the target everywhere you have not matched it. No $\exp$, no saturation; descent rolls to the bottom ($F =$ SVD of $G$).

**Subtle but crucial:** the Gram loss stays well-conditioned *even though the spectrum is flat*. A flat spectrum ($\sigma_k\approx\sigma_{k+1}$) does **not** flatten the Gram loss — it only makes the *choice of the $k$-th mode* slightly ambiguous (the $\sim0.02$ residual of §11). Two different "flatnesses":

| flatness | flat spectrum + contrastive ($\exp$) | flat spectrum + Gram (quadratic) |
|---|---|---|
| effect on the loss | whole loss flat $\to$ **no learning** | loss well-shaped $\to$ **learns the SVD** |
| what is left | total stall (end-to-end) | harmless $0.02$ mode-picking wobble (§11) |

**Structural bonus:** imit-Gram keeps the operator fixed, so it only trains the **encoder** against a fixed target — a standard supervised regression. End-to-end must backpropagate through the whole kernel $\to$ matrix-product $\to$ contrastive chain (the ill-conditioned part imit-Gram never touches).

# Part III — The findings (and why)

## 11. imit-Gram matches fixed MDT — but not *exactly* ($\sim0.02$)
imit-Gram reaches the **global optimum** of its Gram loss (its loss equals the analytic rank-$k$
truncation floor, ratio $1.00\times$) — so it is **not underfit**. Yet its AMI lands $\sim0.02$–$0.06$ off
fixed MDT. Reason: the **flat spectrum**. The singular values near the $k$-th boundary are nearly equal
($\sigma_k\approx\sigma_{k+1}$), so the **top-$k$ subspace is not unique** — a family of rank-$k$
embeddings are equally Gram-optimal, and raw-SVD / $\sqrt{\pi}$-SVD / the neural encoder each pick a
different one, differing by $\sim0.02$. Not a failure — intrinsic ambiguity of truncating a flat spectrum.


## 12. A bigger embedding dimension *hurts* (the "mirage")
The embedding keeps the top-$d$ singular coordinates, each scaled by $\sigma_i$. A fast-decaying spectrum
would make coordinates beyond $k$ tiny and harmless — but the spectrum is **flat**, so coordinates
$k{+}1,\dots,d$ carry **noise, not signal**, and k-means degrades. Bigger $d$ $\Rightarrow$ **lower** AMI.
The next cell shows both §11 (near-tied singular values, eigengap $\approx1$) and §12 (AMI falls with $d$).


In [ ]:
W = transition(Xv[2], knn)                                        # a strong MSRC view
U, s, Vt = svd(W)
print("singular values 2..12 (dropping trivial 1st):", np.round(s[1:12], 3), " <- nearly equal = FLAT")
print(f"eigengap at k={k}:  s[k]/s[k+1] = {s[k]/s[k+1]:.3f}   (~1 => near-degenerate => top-k NOT unique -> S11)")
print("\nAMI vs embedding dimension d (always clustering into k):")
for d in (k, 16, 32, 64):
    E = U[:, 1:d+1] * s[1:d+1]
    print(f"   d={d:3d}:  AMI = {AMI(y, KMeans(k, n_init=10, random_state=0).fit_predict(E)):.3f}")
print("   -> flat tail: extra dims are noise -> AMI FALLS as d grows (S12).")


## 13. Sparse kNN vs dense kernel: a wash — but *not* because they're identical
The kernel can be **dense** (all $N^2$ entries) or **sparse (kNN)**. MDT uses sparse; the collapsing
end-to-end model used dense — so we asked if the collapse was a dense-kernel artifact. Toggling only the
kNN mask (everything else fixed): the AMI is a **wash**. **But the kernels are not interchangeable in
general:** on low-dim views the tight bandwidth makes the dense Gaussian $\approx0$ off-neighbourhood
(mask keeps $\approx100\%$ of the mass); on high-dim views the mask drops real mass ($\sim5$–$9\%$ kept).
That it is a wash **even though the kernels genuinely differ** is the point: the collapse is
**objective-driven, not kernel-driven**. The cell shows the per-view mass kept on MSRC.


In [ ]:
print("view   dim    mass kept by kNN mask    max dropped entry")
for vi, Xview in enumerate(Xv):
    Dm = squareform(pdist(Xview)); n = len(Xview)
    bw = max(row[row > 0].min() for row in Dm)
    Kd = np.exp(-Dm**2 / bw); kn = int(np.floor(np.log(n)))
    Ks = np.zeros_like(Kd)
    for i in range(n):
        nn = np.argsort(Dm[i])[:kn+1]; Ks[i, nn] = Kd[i, nn]
    print(f"  {vi}    {Xview.shape[1]:4d}     {Ks.sum()/Kd.sum():6.1%}                  {Kd[Ks==0].max():.1e}")
print("\n-> low-dim views: dense == sparse (~100% kept); high-dim views: they DIFFER. Yet toggling the")
print("   mask barely changes clustering => collapse is OBJECTIVE-driven, not kernel-driven.")


## 14. The Internal Quality index: picking a trajectory without labels
MDT must choose a **trajectory** (which views, in what order) with no labels, so it needs an
**Internal-Quality (IQ) index** — a score from the operator alone that predicts clustering quality. MDT's
index (Eq. 19–20) is a **contrastive NLL**: softmax each operator row, reward trajectories under which
each point's kNN neighbours have high transition probability,

$$ Q(a,t)=\sum_v\lambda_v\sum_i\sum_{j\in\text{nbr}(i)}-\log\frac{e^{W_{ij}}}{\sum_{x'}e^{W_{ix'}}} . $$

It is **weak** — for the same uniform-logit reason as §10, the softmax barely varies, so $Q$ hardly
distinguishes good trajectories. A **spectral-energy** index — the fraction of singular-value energy in
the top-$k$ modes — is a better *ranker*.

### The result table (paper Table 9)

We rank a pool of trajectories by each label-free index and compare to the true clustering AMI, using two
metrics:
- **Spearman $\rho$** = does the index *rank* trajectories in the true-AMI order? $+1$ perfect, $0$
  unrelated, negative = backwards.
- **selection regret** = if you *deploy* the index's single top pick, how much AMI do you lose vs the best
  in the pool ($\text{oracle}-\text{selected}$)? $0$ = picked the best; lower is better.

| Dataset | $\rho$: $Q_c$ | $\rho$: sil | $\rho$: **en** | regret: $Q_c$ | regret: sil | regret: **en** |
|---|---|---|---|---|---|---|
| MSRC-v5        | $-.25$ | $-.17$ | $\mathbf{.49}$ | $.072$ | $.097$ | $\mathbf{.043}$ |
| Yale           | $-.12$ | $-.59$ | $.11$          | $.073$ | $.131$ | $\mathbf{.064}$ |
| BBCSport       | $.13$  | $-.04$ | $\mathbf{.60}$ | $.125$ | $.164$ | $\mathbf{.019}$ |
| Caltech101-7   | $-.12$ | $.11$  | $\mathbf{.36}$ | $.047$ | $.467$ | $\mathbf{.043}$ |
| Handwritten    | $.24$  | $.05$  | $.19$          | $\mathbf{.050}$ | $.365$ | $.209$ |
| UCI            | $-.28$ | $-.30$ | $\mathbf{.36}$ | $.369$ | $.404$ | $\mathbf{.121}$ |
| Wikipedia-test | $.33$  | $.61$  | $-.47$         | $\mathbf{.000}$ | $\mathbf{.000}$ | $.315$ |
| **mean**       | $-.01$ | $-.05$ | $\mathbf{.23}$ | $.105$ | $.233$ | $.116$ |

**How to read it**
- **MSRC, BBCSport, Caltech, UCI:** spectral energy (**en**) has the highest $\rho$ *and* lowest regret —
  it ranks and selects best.
- **Wikipedia-test — the exception:** en *fails* ($\rho=-.47$, regret $.315$) while $Q_c$ and silhouette
  nail it (regret $.000$). This is the heterogeneous-view case (one near-useless view) — the spectral
  index gets fooled.
- **Handwritten:** en ranks positively ($\rho=.19$) but its single top pick is unlucky (regret $.209$):
  a good ranker, a poor argmax on this set.
- **Mean row:** on $\rho$, **en $+.23$ is the only positive** ($Q_c$ and silhouette $\approx0$ =
  uncorrelated with quality). On regret, **en is lowest on $5/7$** datasets; its *mean* regret ($.116$)
  only ties $Q_c$ ($.105$) because the single Wikipedia failure drags the average up.

**Verdict:** spectral energy is the best *average* ranker/selector, but **not universal** (fails on
heterogeneous views), and the gain is **classical, not DDM** (the $\sqrt{\pi}$ weighting is inert). MDT's
own contrastive $Q_c$ is essentially useless as a ranker here.

The cell reproduces the *per-dataset noise* on three small sets (single pool, illustrative) — note how
silhouette swings wildly, which is why only the 7-dataset average above is trustworthy.


#### The two metrics, precisely

Take a pool of candidate trajectories $\{W_j\}_{j=1}^{M}$. Each has a *true* clustering quality $\text{AMI}(W_j)$ (label-based — the "oracle") and a label-free score $\text{index}(W_j)$ (one of $Q_c$, silhouette, energy).

**Spearman rank correlation $\rho$** — does the index *order* the trajectories the way their true AMI does?

$$ \rho \;=\; 1 - \frac{6\sum_{j=1}^{M} d_j^2}{M\,(M^2-1)}, \qquad d_j = \operatorname{rank}_{\text{index}}(W_j) - \operatorname{rank}_{\text{AMI}}(W_j), $$

where the ranks are $1,\dots,M$ under each ordering. ($\rho$ is just Pearson's correlation of the two rank-lists; the constant $6$ normalises it so identical order $\to +1$, reversed order $\to -1$, unrelated $\to 0$.)

**Selection regret** — how much AMI you lose by *deploying the index's single top pick*:

$$ \text{regret} \;=\; \max_{j}\,\text{AMI}(W_j) \;-\; \text{AMI}\big(W_{j^\star}\big), \qquad j^\star = \arg\max_{j}\,\text{index}(W_j). $$

It is $\ge 0$ (you cannot beat the best in the pool), $=0$ iff the index's favourite $j^\star$ *is* the best trajectory, and lower is better (units = AMI).

**The difference:** $\rho$ scores the *whole ordering*; regret scores *only the $\#1$ pick* ($j^\star$). An index can rank well overall (high $\rho$) yet have a slightly-off favourite (regret $>0$, e.g. Handwritten), or point at a dud (both bad, e.g. Wikipedia) — so both are reported.

### Why the contrastive index $Q$ is weak — and the spectral-energy formula

**Why $Q$ is weak.** To *rank* trajectories, an index must give clearly *different* scores to good vs bad ones. $Q$ gives nearly the *same* score to all of them, so ranking by it is almost random ($\rho\approx 0$ with the true AMI; measured aggregate $\rho\approx+0.06$, essentially uncorrelated).

*Why $Q$ barely changes across trajectories:* it is built from $\text{softmax}(W)$, but the diffused operator has near-uniform tiny rows ($W_{ij}\approx 1/N$). Then $\exp(W_{ij})\approx 1$ for **every** $j$, so the softmax is $\approx$ uniform — every target, including the kNN neighbours $Q$ rewards, gets probability $\approx 1/N$ **regardless of the trajectory**. Hence

$$ Q \;\approx\; (\#\text{neighbours})\times\big(-\log\tfrac{1}{N}\big) \;=\; (\#\text{neighbours})\times\log N \;\approx\; \text{const} $$

for *every* trajectory. A near-constant score cannot rank anything — that is the "weak."

*Concretely* ($N=200,\ knn=5$): the softmax is $\approx$ uniform, so a neighbour's probability is $\approx 1/200 = 0.005$ and its NLL is $\approx -\log 0.005 = 5.3$. A *good* trajectory might push it to $0.006$ (NLL $5.1$), a *bad* one to $0.004$ (NLL $5.5$) — a tiny $0.4$ spread, and not even reliably aligned with clustering. Every trajectory reads $Q\approx 5.3$; you cannot pick the good one out of the pile.

**Why spectral energy is *not* weak.** It reads the **singular values** of $W$, which *do* vary meaningfully between trajectories (a good one concentrates energy in its top-$k$ modes, a bad one spreads it out), so it can rank. In short: $\exp(\cdot)$ on near-uniform rows **erases** the contrast $Q$ needs, while the singular-value spectrum **keeps** it. Its formula:

$$ \text{energy}(W) \;=\; \frac{\sum_{i=2}^{k+1}\sigma_i^2}{\sum_{i\ge 2}\sigma_i^2}, $$

where $\sigma_1\ge\sigma_2\ge\cdots$ are the singular values of $W$:
- $\sigma_1$ = the **trivial mode** (stationary/constant, $\approx 1$) — dropped, so the sums start at $i=2$;
- numerator = energy in the $k$ modes the embedding uses ($i=2,\dots,k+1$); denominator = all non-trivial energy.

Computed cheaply via $\sum_i\sigma_i^2 = \lVert W\rVert_F^2 = \sum_{ij}W_{ij}^2$, so the denominator is just $\sum_{ij}W_{ij}^2 - \sigma_1^2$ (only the top few singular values are needed).

**Eckart–Young link:** the *dropped* energy is exactly the truncation error $\lVert W-W_k\rVert_F^2=\sum_{i>k}\sigma_i^2$, so $\text{energy} = 1 - (\text{relative truncation error})$ — a trajectory scores high exactly when its operator is cleanly summarised by the top-$k$ SVD, the very embedding you will use. *(Sibling **gap** index: $\sigma_{k+1}/\sigma_{k+2}$, how sharply the spectrum drops past the kept modes; energy was the stronger of the two.)*

#### Reading the formula: why $\sigma^2$, and why "part over whole"

**Why $\sigma_i^2$ and not $\sigma_i$.** Only the *squares* add up to the whole matrix. There is an identity
$$ \lVert W\rVert_F^2 = \sum_{ij}W_{ij}^2 = \sigma_1^2+\sigma_2^2+\sigma_3^2+\cdots, $$
so each $\sigma_i^2$ is genuinely "mode $i$'s share of the whole," and the shares sum to $100\%$. The raw values $\sigma_1+\sigma_2+\cdots$ (the *nuclear norm*) equal no natural "total," so a ratio built from them would mean nothing. Three ways to see it:
- **A fraction needs a whole that adds up:** squares do ($\sum_i\sigma_i^2=\lVert W\rVert_F^2$); raw $\sigma$ don't.
- **It equals reconstruction error (Eckart–Young):** dropping the tail costs $\lVert W-W_k\rVert_F^2=\sum_{i>k}\sigma_i^2$ (squared), so $\text{energy}=1-\text{error}/\text{total}$ works *only* with squares.
- **It is what "energy" means:** in physics / signal processing, energy $=$ amplitude$^2$. Squaring also right-weights the modes — a $\sigma=0.9$ mode carries $0.81$, a $\sigma=0.1$ mode only $0.01$, so weak/noise modes count almost nothing.

**Why the numerator runs $i=2$ to $k+1$ but the denominator over all $i\ge 2$.** A ratio is *part $\div$ whole*:
- **Numerator = the part you keep** — the $k$ modes the embedding actually uses. After dropping the trivial $\sigma_1$, those are indices $2,3,\dots,k+1$ (exactly $k$ terms; the "$+1$" is just the shift from having dropped $\sigma_1$ — without that drop it would read "$1$ to $k$").
- **Denominator = the whole** — *all* non-trivial modes, $i=2,\dots,N$ ($N=$ number of points, finite; "$i\ge 2$" just means "all the rest"). That is the kept modes **plus** every dropped mode.

So the fraction asks: *"of all the operator's content, how much sits in the $k$ modes I keep?"* — numerator a **subset** ($k$ modes), denominator the **total** (all modes). Both start at $i=2$ because the trivial mode $\sigma_1$ is excluded from *both* (it carries no cluster info, so it must not inflate either the part or the whole).

**Concrete** ($k=2$, non-trivial $\sigma=[0.9,\,0.8,\,0.1,\,0.05,\dots]$):
- numerator $= 0.9^2+0.8^2 = 1.45$  (the $2$ modes kept),
- denominator $= 1.45 + 0.1^2 + 0.05^2 + \cdots \approx 1.46$  (all modes),
- energy $= 1.45/1.46 \approx 0.99$ → "99% of the content is in the 2 modes I keep."

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score

def rho_of_indices(name):
    mm = scipy.io.loadmat(f"/tmp/Multi-view-datasets/{name}.mat")
    Xs = [np.asarray(v).astype(float) for v in mm["X"].ravel()]; yy = np.asarray(mm["y"]).ravel()
    kk = len(np.unique(yy)); kn = max(2, int(np.floor(np.log(len(yy)))))
    Pw = [transition(v, kn) for v in Xs]
    masks = [(p > 0) & ~np.eye(len(p), dtype=bool) for p in Pw]
    r = np.random.default_rng(0); amis, cq, sil, en = [], [], [], []
    for _ in range(25):
        seq = r.integers(0, len(Pw), 6)
        W = reduce(lambda A, B: B @ A, [Pw[i] for i in seq])
        U, s, _ = svd(W); E = U[:, 1:kk+1] * s[1:kk+1]
        lab = KMeans(kk, n_init=5, random_state=0).fit_predict(E)
        amis.append(AMI(yy, lab)); sil.append(silhouette_score(E, lab) if len(set(lab)) > 1 else -1.0)
        ex = np.exp(np.clip(W, -20, 20)); np.fill_diagonal(ex, 0.0)
        lp = np.log(np.clip(ex / ex.sum(1, keepdims=True), 1e-12, None))
        cq.append(-np.mean([-lp[msk].sum() for msk in masks]) / len(W))
        en.append((s[1:kk+1]**2).sum() / max((s**2).sum() - s[0]**2, 1e-12))
    f = lambda z: spearmanr(z, amis).correlation
    return f(cq), f(sil), f(en)

print(f"{'dataset':11s} {'rho Q_c':>8s} {'rho sil':>8s} {'rho en':>8s}   (single 25-trajectory pool)")
for name in ["MSRC-v5", "Yale", "BBCSport"]:
    a, b, c = rho_of_indices(name); print(f"{name:11s} {a:>8.2f} {b:>8.2f} {c:>8.2f}")
print("\n-> single pools are NOISY (silhouette especially); only the 7-dataset average (table above) is")
print("   trustworthy: en +.23 (only positive), silhouette -.05, contrastive Q_c -.01.")


# Part IV — One root cause: the flat spectrum

The diffused operator's singular values are nearly equal (its rows are $\approx$ uniform). That single
property explains every finding above:

| flat spectrum $\Rightarrow$ | consequence | section |
|---|---|---|
| Gram optimum $=$ the SVD | imit net **matches** fixed MDT | 8–10 |
| $\sigma_k\approx\sigma_{k+1}$ (top-$k$ non-unique) | imit $\ne$ fixed by $\sim0.02$ | 11 |
| tail singular values not small | bigger embedding dim = **worse** | 12 |
| $\exp(W)$ logits $\approx$ uniform | end-to-end contrastive **stalls/collapses**; $Q_c$ weak | 10, 14 |
| stationary $\pi\approx$ uniform | DDM's $\sqrt{\pi}$ weighting inert (gains are classical) | 13, 14 |

**Bottom line.** MDT builds the operator (kernel + fusion + trajectory); its embedding is the truncated
SVD; Eckart–Young makes that SVD the best rank-$k$ summary; DDM's network can only *match* it (imit-Gram)
and *fails* when it tries to learn the operator itself (end-to-end). The genuine improvements are all
**classical operator choices** — never the neural machinery.
